<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/_retrain_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive

drive.mount('/content/drive')

annotations = """Shoplifting001_x264.mp4 Shoplifting 1550 2000 -1 -1
Shoplifting004_x264.mp4 Shoplifting 2200 4900 -1 -1
Shoplifting005_x264.mp4 Shoplifting 720 930 -1 -1
Shoplifting007_x264.mp4 Shoplifting 550 760 4630 4920
Shoplifting010_x264.mp4 Shoplifting 750 920 1550 1970
Shoplifting015_x264.mp4 Shoplifting 2010 2160 -1 -1
Shoplifting016_x264.mp4 Shoplifting 630 720 -1 -1
Shoplifting017_x264.mp4 Shoplifting 360 420 -1 -1
Shoplifting020_x264.mp4 Shoplifting 2340 2460 -1 -1
Shoplifting021_x264.mp4 Shoplifting 2070 2220 -1 -1
Shoplifting022_x264.mp4 Shoplifting 270 420 1440 1560
Shoplifting027_x264.mp4 Shoplifting 1080 1160 1470 1710
Shoplifting028_x264.mp4 Shoplifting 570 840 -1 -1
Shoplifting029_x264.mp4 Shoplifting 1020 1470 -1 -1
Shoplifting031_x264.mp4 Shoplifting 120 330 -1 -1
Shoplifting033_x264.mp4 Shoplifting 630 750 -1 -1
Shoplifting034_x264.mp4 Shoplifting 7350 7470 -1 -1
Shoplifting037_x264.mp4 Shoplifting 1140 1200 -1 -1
Shoplifting039_x264.mp4 Shoplifting 2190 2340 -1 -1
Shoplifting044_x264.mp4 Shoplifting 11070 11250 -1 -1
Shoplifting049_x264.mp4 Shoplifting 1020 1350 -1 -1"""

output_path = "/content/drive/MyDrive/shoplifting/shoplifting_annotations.txt"
with open(output_path, 'w') as f:
    f.write(annotations)

print("Annotation file created successfully.")

# Verify
with open(output_path, 'r') as f:
    lines = f.readlines()
print(f"Total lines written: {len(lines)}")
print(f"First line: {lines[0].strip()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Annotation file created successfully.
Total lines written: 21
First line: Shoplifting001_x264.mp4 Shoplifting 1550 2000 -1 -1


In [7]:
import cv2
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

# ─── PATHS ───
shoplifting_video_path = "/content/drive/MyDrive/shoplifting/ucf_shoplifting/shoplifting_data"
normal_video_path = "/content/drive/MyDrive/shoplifting/ucf_normal/normal_video"
annotation_file = "/content/drive/MyDrive/shoplifting/shoplifting_annotations.txt"

output_shoplifting = "/content/ucf_clips/shoplifting"
output_normal = "/content/ucf_clips/normal"
os.makedirs(output_shoplifting, exist_ok=True)
os.makedirs(output_normal, exist_ok=True)

PADDING = 30

# ─── STEP 1: Parse annotations ───
annotations = {}
with open(annotation_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        print(f"Parsing line: {parts}")  # debug
        if len(parts) >= 6:
            filename = parts[0]
            start1, end1 = int(parts[2]), int(parts[3])
            start2 = int(parts[4]) if parts[4] != '-1' else -1
            end2 = int(parts[5]) if parts[5] != '-1' else -1
            annotations[filename] = [(start1, end1)]
            if start2 != -1:
                annotations[filename].append((start2, end2))

print(f"\nLoaded annotations for {len(annotations)} videos\n")

# ─── STEP 2: Extract shoplifting clips ───
def extract_clip(video_path, start_frame, end_frame, output_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    start = max(0, start_frame - PADDING)
    end = min(total_frames, end_frame + PADDING)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    for _ in range(end - start):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)

    cap.release()
    out.release()

total_shoplifting = 0

for filename, segments in annotations.items():
    video_path = os.path.join(shoplifting_video_path, filename)

    if not os.path.exists(video_path):
        print(f"  Skipping {filename} — not found")
        continue

    for i, (start, end) in enumerate(segments):
        out_name = f"ucf_shoplifting_{total_shoplifting}.mp4"
        out_path = os.path.join(output_shoplifting, out_name)
        extract_clip(video_path, start, end, out_path)
        total_shoplifting += 1
        print(f"  Extracted: {filename} segment {i+1}")

print(f"\nShoplifting clips extracted: {total_shoplifting}")

# ─── STEP 3: Extract normal clips ───
normal_files = [f for f in os.listdir(normal_video_path) if f.endswith('.mp4')]
total_normal = 0
CLIP_LENGTH = 150

for filename in normal_files:
    video_path = os.path.join(normal_video_path, filename)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    starts = range(0, total_frames - CLIP_LENGTH, CLIP_LENGTH)

    for start in starts:
        out_name = f"ucf_normal_{total_normal}.mp4"
        out_path = os.path.join(output_normal, out_name)
        extract_clip(video_path, start, start + CLIP_LENGTH, out_path)
        total_normal += 1

print(f"Normal clips extracted: {total_normal}")

# ─── STEP 4: Save to Drive ───
print("\nSaving to Drive...")
drive_clips = "/content/drive/MyDrive/shoplifting/ucf_clips"

if os.path.exists(drive_clips):
    shutil.rmtree(drive_clips)

shutil.copytree("/content/ucf_clips", drive_clips)
print(f"Saved to Drive → {drive_clips}")
print(f"\nTotal shoplifting clips: {total_shoplifting}")
print(f"Total normal clips: {total_normal}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Parsing line: ['Shoplifting001_x264.mp4', 'Shoplifting', '1550', '2000', '-1', '-1']
Parsing line: ['Shoplifting004_x264.mp4', 'Shoplifting', '2200', '4900', '-1', '-1']
Parsing line: ['Shoplifting005_x264.mp4', 'Shoplifting', '720', '930', '-1', '-1']
Parsing line: ['Shoplifting007_x264.mp4', 'Shoplifting', '550', '760', '4630', '4920']
Parsing line: ['Shoplifting010_x264.mp4', 'Shoplifting', '750', '920', '1550', '1970']
Parsing line: ['Shoplifting015_x264.mp4', 'Shoplifting', '2010', '2160', '-1', '-1']
Parsing line: ['Shoplifting016_x264.mp4', 'Shoplifting', '630', '720', '-1', '-1']
Parsing line: ['Shoplifting017_x264.mp4', 'Shoplifting', '360', '420', '-1', '-1']
Parsing line: ['Shoplifting020_x264.mp4', 'Shoplifting', '2340', '2460', '-1', '-1']
Parsing line: ['Shoplifting021_x264.mp4', 'Shoplifting', '2070', '2220', '-1', '-1']
Parsing line: ['Shoplif

In [9]:
!pip install ultralytics
import os
import json
import shutil
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

model = YOLO("yolo11n-pose.pt")

# ─── PATHS ───
new_clips = {
    "/content/drive/MyDrive/shoplifting/ucf_clips/shoplifting": 1,
    "/content/drive/MyDrive/shoplifting/ucf_clips/normal": 0,
}

drive_keypoints = "/content/drive/MyDrive/shoplifting/keypoints_normalized"
output_dir = "/content/keypoints_normalized"

# Load existing keypoints count
os.makedirs(output_dir, exist_ok=True)
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) == 0:
    print("Copying existing keypoints from Drive...")
    shutil.copytree(drive_keypoints, output_dir, dirs_exist_ok=True)

existing_count = len(os.listdir(output_dir))
print(f"Existing keypoints: {existing_count}")

SAMPLE_EVERY = 3
MAX_FRAMES = 50
LEFT_HIP, RIGHT_HIP = 11, 12
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6

def normalize_keypoints(kps):
    left_hip = kps[LEFT_HIP]
    right_hip = kps[RIGHT_HIP]
    center = (left_hip + right_hip) / 2
    left_shoulder = kps[LEFT_SHOULDER]
    right_shoulder = kps[RIGHT_SHOULDER]
    shoulder_center = (left_shoulder + right_shoulder) / 2
    scale = np.linalg.norm(shoulder_center - center)
    if scale < 1e-6:
        scale = 1.0
    normalized = (kps - center) / scale
    return normalized.flatten().tolist()

def extract_keypoints_normalized(video_path):
    cap = cv2.VideoCapture(video_path)
    keypoint_sequence = []
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % SAMPLE_EVERY == 0:
            results = model(frame, verbose=False)
            if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
                kps = results[0].keypoints.xy[0].cpu().numpy()
                if kps.shape[0] == 17:
                    normalized = normalize_keypoints(kps)
                    keypoint_sequence.append(normalized)
                else:
                    keypoint_sequence.append([0] * 34)
            else:
                keypoint_sequence.append([0] * 34)
            if len(keypoint_sequence) >= MAX_FRAMES:
                break
        frame_idx += 1
    cap.release()
    return keypoint_sequence

total = existing_count
skipped = 0

for folder_path, label in new_clips.items():
    files = [f for f in os.listdir(folder_path) if f.endswith('.mp4')]
    print(f"\nProcessing: {folder_path.split('/')[-1]} — {len(files)} clips")

    for filename in files:
        video_path = os.path.join(folder_path, filename)
        sequence = extract_keypoints_normalized(video_path)

        if len(sequence) < 10:
            skipped += 1
            continue

        while len(sequence) < MAX_FRAMES:
            sequence.append([0] * 34)
        sequence = sequence[:MAX_FRAMES]

        output = {
            "file": filename,
            "label": label,
            "keypoints": sequence
        }

        out_path = os.path.join(output_dir, f"{label}_{total}.json")
        with open(out_path, 'w') as f:
            json.dump(output, f)

        total += 1
        if total % 50 == 0:
            print(f"  Processed {total} clips...")

print(f"\nExtraction done.")
print(f"Previous: {existing_count} | New total: {total} | Skipped: {skipped}")

# ─── Save merged keypoints to Drive ───
print("\nSaving merged keypoints to Drive...")
if os.path.exists(drive_keypoints):
    shutil.rmtree(drive_keypoints)
shutil.copytree(output_dir, drive_keypoints)
print(f"Saved → {drive_keypoints}")
print(f"Total keypoint files: {len(os.listdir(drive_keypoints))}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying existing keypoints from Drive...
Existing keypoints: 1037

Processing: shoplifting — 20 clips
  Processed 1050 clips...

Processing: normal — 351 clips
  Processed 1100 clips...
  Processed 1150 clips...
  Processed 1200 clips...
  Processed 1250 clips...
  Processed 1300 clips...
  Processed 1350 clips...
  Processed 1400 clips...

Extraction done.
Previous: 1037 | New total: 1408 | Skipped:

In [10]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import shutil
from google.colab import drive

drive.mount('/content/drive')

# ─── STEP 1: Load merged keypoints ───
drive_keypoints = "/content/drive/MyDrive/shoplifting/keypoints_normalized"
local_keypoints = "/content/keypoints_normalized"

if not os.path.exists(local_keypoints):
    print("Copying from Drive...")
    shutil.copytree(drive_keypoints, local_keypoints)
    print("Done.")

print("Loading keypoint files...")
X, y = [], []

for file in os.listdir(local_keypoints):
    if file.endswith('.json'):
        with open(os.path.join(local_keypoints, file)) as f:
            data = json.load(f)
        X.append(data['keypoints'])
        y.append(data['label'])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"Loaded: {X.shape}")
print(f"Shoplifting: {int(y.sum())} | Normal: {int((y==0).sum())}")

# ─── STEP 2: Normalize ───
X_mean = X.mean()
X_std = X.std()
X = (X - X_mean) / (X_std + 1e-8)

# ─── STEP 3: Train/val split ───
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train)} | Val: {len(X_val)}")

# ─── STEP 4: Dataset ───
class KeypointDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = KeypointDataset(X_train, y_train)
val_ds   = KeypointDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32)

# ─── STEP 5: Bidirectional LSTM ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256,
                 num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing: {device}")

model = ShopliftingLSTM().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=4, factor=0.5
)

# ─── STEP 6: Training ───
EPOCHS = 50
best_val_acc = 0

print("\nTraining...\n")
for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        correct += ((preds > 0.5) == yb).sum().item()
        total += len(yb)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            val_loss += criterion(preds, yb).item()
            val_correct += ((preds > 0.5) == yb).sum().item()
            val_total += len(yb)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_dl):.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/vigiq_best_v3.pth')
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

# ─── STEP 7: Evaluation ───
model.load_state_dict(torch.load('/content/vigiq_best_v3.pth'))
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(device)
        preds = model(xb)
        all_preds.extend((preds > 0.5).cpu().numpy())
        all_labels.extend(yb.numpy())

print("\n=== RESULTS v3 ===")
print(classification_report(all_labels, all_preds,
      target_names=['Normal', 'Shoplifting']))
print(f"Best Val Accuracy: {best_val_acc:.4f}")

# ─── STEP 8: Save to Drive ───
torch.save(model.state_dict(),
           '/content/drive/MyDrive/shoplifting/vigiq_best_v3.pth')
np.save('/content/drive/MyDrive/shoplifting/X_mean_v3.npy',
        np.array([X_mean]))
np.save('/content/drive/MyDrive/shoplifting/X_std_v3.npy',
        np.array([X_std]))
print("\nModel v3 saved to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading keypoint files...
Loaded: (1408, 50, 34)
Shoplifting: 436 | Normal: 972

Train: 1126 | Val: 282

Using: cuda

Training...

Epoch 01/50 | Train Loss: 0.6457 | Train Acc: 0.6306 | Val Acc: 0.6915
  ✓ Best model saved (val_acc: 0.6915)
Epoch 02/50 | Train Loss: 0.6339 | Train Acc: 0.6892 | Val Acc: 0.6915
Epoch 03/50 | Train Loss: 0.6126 | Train Acc: 0.6901 | Val Acc: 0.6915
Epoch 04/50 | Train Loss: 0.5798 | Train Acc: 0.6936 | Val Acc: 0.5993
Epoch 05/50 | Train Loss: 0.5740 | Train Acc: 0.6545 | Val Acc: 0.6915
Epoch 06/50 | Train Loss: 0.5562 | Train Acc: 0.6901 | Val Acc: 0.6915
Epoch 07/50 | Train Loss: 0.5823 | Train Acc: 0.6901 | Val Acc: 0.6915
Epoch 08/50 | Train Loss: 0.5684 | Train Acc: 0.6901 | Val Acc: 0.6915
Epoch 09/50 | Train Loss: 0.5580 | Train Acc: 0.6874 | Val Acc: 0.6915
Epoch 10/50 | Train Loss: 0.5302 | Train Acc: 0.7176 | Val Acc

In [11]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import shutil
from google.colab import drive

drive.mount('/content/drive')

# ─── STEP 1: Load merged keypoints ───
drive_keypoints = "/content/drive/MyDrive/shoplifting/keypoints_normalized"
local_keypoints = "/content/keypoints_normalized"

if not os.path.exists(local_keypoints):
    print("Copying from Drive...")
    shutil.copytree(drive_keypoints, local_keypoints)
    print("Done.")

print("Loading keypoint files...")
X, y = [], []

for file in os.listdir(local_keypoints):
    if file.endswith('.json'):
        with open(os.path.join(local_keypoints, file)) as f:
            data = json.load(f)
        X.append(data['keypoints'])
        y.append(data['label'])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"Loaded: {X.shape}")
print(f"Shoplifting: {int(y.sum())} | Normal: {int((y==0).sum())}")

# ─── STEP 2: Normalize ───
X_mean = X.mean()
X_std = X.std()
X = (X - X_mean) / (X_std + 1e-8)

# ─── STEP 3: Train/val split ───
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train)} | Val: {len(X_val)}")

# ─── STEP 4: Dataset ───
class KeypointDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = KeypointDataset(X_train, y_train)
val_ds   = KeypointDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32)

# ─── STEP 5: Bidirectional LSTM with class weighting ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256,
                 num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)  # no Sigmoid — BCEWithLogitsLoss handles it
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing: {device}")

model = ShopliftingLSTM().to(device)

# Class weight — penalize missing shoplifting more
normal_count = int((y == 0).sum())
shoplifting_count = int(y.sum())
pos_weight = torch.tensor([normal_count / shoplifting_count]).to(device)
print(f"Class weight: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=4, factor=0.5
)

# ─── STEP 6: Training ───
EPOCHS = 50
best_val_acc = 0

print("\nTraining...\n")
for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        correct += ((torch.sigmoid(preds) > 0.5) == yb).sum().item()
        total += len(yb)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            val_loss += criterion(preds, yb).item()
            val_correct += ((torch.sigmoid(preds) > 0.5) == yb).sum().item()
            val_total += len(yb)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_dl):.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/vigiq_best_v4.pth')
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

# ─── STEP 7: Evaluation ───
model.load_state_dict(torch.load('/content/vigiq_best_v4.pth'))
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(device)
        preds = model(xb)
        all_preds.extend((torch.sigmoid(preds) > 0.5).cpu().numpy())
        all_labels.extend(yb.numpy())

print("\n=== RESULTS v4 ===")
print(classification_report(all_labels, all_preds,
      target_names=['Normal', 'Shoplifting']))
print(f"Best Val Accuracy: {best_val_acc:.4f}")

# ─── STEP 8: Save to Drive ───
torch.save(model.state_dict(),
           '/content/drive/MyDrive/shoplifting/vigiq_best_v4.pth')
np.save('/content/drive/MyDrive/shoplifting/X_mean_v4.npy',
        np.array([X_mean]))
np.save('/content/drive/MyDrive/shoplifting/X_std_v4.npy',
        np.array([X_std]))
print("\nModel v4 saved to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading keypoint files...
Loaded: (1408, 50, 34)
Shoplifting: 436 | Normal: 972

Train: 1126 | Val: 282

Using: cuda
Class weight: 2.23

Training...

Epoch 01/50 | Train Loss: 0.9437 | Train Acc: 0.4281 | Val Acc: 0.4007
  ✓ Best model saved (val_acc: 0.4007)
Epoch 02/50 | Train Loss: 0.8952 | Train Acc: 0.4618 | Val Acc: 0.4610
  ✓ Best model saved (val_acc: 0.4610)
Epoch 03/50 | Train Loss: 0.9053 | Train Acc: 0.4964 | Val Acc: 0.4504
Epoch 04/50 | Train Loss: 0.9053 | Train Acc: 0.4849 | Val Acc: 0.4397
Epoch 05/50 | Train Loss: 0.8903 | Train Acc: 0.5018 | Val Acc: 0.5000
  ✓ Best model saved (val_acc: 0.5000)
Epoch 06/50 | Train Loss: 0.8828 | Train Acc: 0.5266 | Val Acc: 0.4858
Epoch 07/50 | Train Loss: 0.9217 | Train Acc: 0.5062 | Val Acc: 0.5106
  ✓ Best model saved (val_acc: 0.5106)
Epoch 08/50 | Train Loss: 0.9698 | Train Acc: 0.4529 | Val Acc: 0.47